In [20]:
%pip install "requests[socks]"
%pip install selenium


[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 1.2 MB/s eta 0:00:0000:0100:01
Using cached sniffio-1.3.1-py3-none-any.whl (10 kB)
  Attempting uninstall: certifi
    Found existing installation: certifi 2026.7.22
    Uninstalling certifi-2026.7.22:
      Successfully uninstalled certifi-2026.7.22

[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
from pathlib import Path
import socket

In [53]:
output_dir = Path("responses")
output_dir.mkdir(parents=True, exist_ok=True)

for request_number in range(2005,2026):
    print(request_number)
    url = f"https://web.archive.org/cdx/search/cdx?url=https://www.imdb.com/chart/top/&from={request_number}&to={request_number}&output=json&fl=timestamp,original,statuscode,mimetype,digest"

    try:
        response = requests.get(
            url,
            proxies=proxies,
            timeout=60,
        )
        response.raise_for_status()

        output_file = output_dir / f"response_{request_number}.txt"
        output_file.write_bytes(response.content)

        print(f"OK {request_number}: {output_file}")

    except requests.RequestException as e:
        print(f"ERROR :/ {request_number}: {e}")

proxy port is reachable


In [43]:
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import WebDriverException
import time


def getPage(url, folder, exact_date, page_name, group_name, file_extension):

    output_dir = Path(folder)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_file = output_dir / f"{exact_date}_{page_name}.{file_extension}"

    if output_file.exists():
        print(f"SKIP {exact_date}: Already downloaded")
        return True

    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--proxy-server=socks5://127.0.0.1:9050")
    
    driver = webdriver.Chrome(options=options)

    try:
        driver.set_page_load_timeout(90)

        for attempt in range(3):
            try:
                driver.get("https://check.torproject.org/")
                if(driver.title=="Congratulations. This browser is configured to use Tor." and driver.current_url=="https://check.torproject.org/"):
                    print("__TOR is OK")
                driver.get(url)
                time.sleep(3)

                html = driver.page_source

                if (
                    "403 Forbidden" in html
                    or "Error 403" in html
                    or "ERR_TOO_MANY_REDIRECTS" in html
                    or "redirected you too many times" in html
                ):
                    print(f"NOT SCRAPPABLE: {exact_date}")
                    return False
                
                if "403 Forbidden" in html or "Error 403" in html:
                    print(f"403 FORBIDDEN: {exact_date}")
                    return False

                if "429 Too Many Requests" in html:
                    print(f"429 RATE LIMIT: {exact_date}")
                    return None

                if len(html) < 1000:
                    raise Exception("Page too short")

                output_file.write_text(html, encoding="utf-8")

                print(f"OK {exact_date}: {output_file}")
                return True

            except Exception as e:
                print(f"Attempt {attempt + 1}/3 failed: {exact_date}: {e}")

                if attempt < 2:
                    time.sleep(10)

        return False

    finally:
        driver.quit()

In [44]:
import pandas as pd
import json
from uuid import uuid4
from time import sleep
import random
from pathlib import Path

df = pd.DataFrame(columns=["exact_date"])

SAVE_TO_FILE = True

host = "127.0.0.1"
port = 9050

try:
    with socket.create_connection((host, port), timeout=5):
        print("proxy port is reachable")
except OSError as e:
    print(f"proxy connection failed: {e}")

proxies = {
    "http": "socks5h://127.0.0.1:9050",
    "https": "socks5h://127.0.0.1:9050",
}

for i in range(2005, 2026):

    with open(f"responses/response_{i}.txt", 'r') as file:
        print(f"\n\n__________________ {i} __________________\n\n")

        parsed_file_as_array = json.load(file)

        latest_per_month = {}

        for content in parsed_file_as_array[1:]:
            date = content[0]
            month = date[:6]

            latest_per_month.setdefault(month, []).append(date)

        folder = Path(f"{i}_top_250")
        folder.mkdir(parents=True, exist_ok=True)

        for month, dates in latest_per_month.items():

            # Skip month if a record already exists
            if any(folder.glob(f"{month}*_top_250.html")):
                print(f"SKIP {month}: Already downloaded")
                continue

            # Try dates from latest to oldest
            dates.sort(reverse=True)

            for date in dates:
                success = getPage(
                    f"https://web.archive.org/web/{date}id_/https://www.imdb.com/chart/top/",
                    f"{i}_top_250",
                    date,
                    "top_250",
                    f"{i}_top_250",
                    "html"
                )

                if success:
                    if SAVE_TO_FILE:
                        df.loc[len(df)] = {"exact_date": date}
                    break

                sleep(random.uniform(1, 4))

if SAVE_TO_FILE:
    df.to_csv(f"responses-{uuid4()}.csv", index=False)

proxy port is reachable


__________________ 2005 __________________


SKIP 200501: Already downloaded
SKIP 200502: Already downloaded
SKIP 200503: Already downloaded
SKIP 200504: Already downloaded
SKIP 200505: Already downloaded
SKIP 200506: Already downloaded
SKIP 200507: Already downloaded
SKIP 200508: Already downloaded
SKIP 200509: Already downloaded
SKIP 200510: Already downloaded
SKIP 200511: Already downloaded
SKIP 200512: Already downloaded


__________________ 2006 __________________


SKIP 200601: Already downloaded
SKIP 200602: Already downloaded
SKIP 200603: Already downloaded
SKIP 200604: Already downloaded
SKIP 200605: Already downloaded
SKIP 200606: Already downloaded
SKIP 200607: Already downloaded
SKIP 200608: Already downloaded
SKIP 200609: Already downloaded
SKIP 200610: Already downloaded
SKIP 200611: Already downloaded
SKIP 200612: Already downloaded


__________________ 2007 __________________


SKIP 200701: Already downloaded
SKIP 200702: Already downloaded
SKI

In [42]:
from pathlib import Path

for folder in Path(".").glob("*_top_250"):
    for file in folder.glob("*_top_250.html"):
        html = file.read_text(encoding="utf-8", errors="ignore")

        if (
            "403 Forbidden" in html
            or "Error 403" in html
            or "ERR_TOO_MANY_REDIRECTS" in html
            or "redirected you too many times" in html
        ):
            file.unlink()
            print(f"DELETED: {file}")

In [46]:
import csv
import json
import re
from pathlib import Path

from bs4 import BeautifulSoup


FIELDS = [
    "snapshot",
    "rank",
    "title",
    "year",
    "imdb_id",
    "title_url",
    "rating",
    "votes",
    "source_file",
]


def title_id(url):
    match = re.search(r"/title/(tt\d+)", url or "")
    return match.group(1) if match else ""


def find_item_lists(obj):
    """Yield itemListElement lists from JSON-LD, including nested @graph data."""
    if isinstance(obj, dict):
        items = obj.get("itemListElement")
        if isinstance(items, list):
            yield items

        for key, value in obj.items():
            if key != "itemListElement":
                yield from find_item_lists(value)

    elif isinstance(obj, list):
        for value in obj:
            yield from find_item_lists(value)


def parse_json_ld(soup, source):
    for script in soup.select('script[type="application/ld+json"]'):
        try:
            data = json.loads(script.string or script.get_text())
        except json.JSONDecodeError:
            continue

        for items in find_item_lists(data):
            rows = []

            for position, entry in enumerate(items, start=1):
                if not isinstance(entry, dict):
                    continue

                movie = entry.get("item", entry)
                if not isinstance(movie, dict):
                    continue

                url = movie.get("url", "")
                rating = movie.get("aggregateRating") or {}

                rows.append({
                    "snapshot": source.stem.split("_")[0],
                    "rank": entry.get("position", position),
                    "title": movie.get("name", ""),
                    "year": "",
                    "imdb_id": title_id(url),
                    "title_url": url,
                    "rating": rating.get("ratingValue", ""),
                    "votes": rating.get("ratingCount", ""),
                    "source_file": str(source),
                })

            if rows:
                return rows[:250]

    return []

def parse_table_rows(soup, source):
    snapshot = source.stem.split("_")[0]

    # Older IMDb pages have a chart table with Rank, Rating, Title, and Votes columns.
    for table in soup.find_all("table"):
        table_rows = table.find_all("tr", recursive=False)
        if not table_rows:
            continue

        header_cells = table_rows[0].find_all(["th", "td"], recursive=False)
        headers = [cell.get_text(" ", strip=True).lower() for cell in header_cells]

        if not {"rank", "rating", "title", "votes"}.issubset(headers):
            continue

        columns = {name: headers.index(name) for name in ("rank", "rating", "title", "votes")}
        rows = []

        for tr in table_rows[1:]:
            cells = tr.find_all(["th", "td"], recursive=False)
            if len(cells) <= max(columns.values()):
                continue

            title_cell = cells[columns["title"]]
            title_link = next(
                (
                    link for link in title_cell.select("a[href]")
                    if title_id(link.get("href", ""))
                ),
                None,
            )
            if title_link is None:
                continue

            rank_text = cells[columns["rank"]].get_text(" ", strip=True)
            rank_match = re.search(r"\d+", rank_text)
            if not rank_match:
                continue

            title_url = title_link.get("href", "")
            title_text = title_cell.get_text(" ", strip=True)
            year_match = re.search(r"\b(?:18|19|20)\d{2}\b", title_text)

            rows.append({
                "snapshot": snapshot,
                "rank": int(rank_match.group()),
                "title": title_link.get_text(" ", strip=True),
                "year": year_match.group() if year_match else "",
                "imdb_id": title_id(title_url),
                "title_url": title_url,
                "rating": cells[columns["rating"]].get_text(" ", strip=True),
                "votes": cells[columns["votes"]].get_text(" ", strip=True),
                "source_file": str(source),
            })

        if rows:
            return sorted(rows, key=lambda row: int(row["rank"]))[:250]

    # Newer pages use IMDb-specific classes and data attributes.
    rows = []

    for tr in soup.select("tr"):
        title_link = next(
            (
                link for link in tr.select("a[href]")
                if title_id(link.get("href", ""))
            ),
            None,
        )
        if title_link is None:
            continue

        title_url = title_link.get("href", "")
        rank_element = tr.select_one('span[name="ir"]')
        if rank_element is None:
            cells = tr.find_all("td", recursive=False)
            rank_element = cells[0] if cells else None

        rank_text = rank_element.get_text(" ", strip=True) if rank_element else ""
        rank_match = re.search(r"\b(\d+)\.?\b", rank_text)
        rank = int(rank_match.group(1)) if rank_match else len(rows) + 1

        year_element = tr.select_one(".secondaryInfo")
        year_match = re.search(
            r"\b(?:18|19|20)\d{2}\b",
            year_element.get_text() if year_element else "",
        )

        rating_element = tr.select_one('span[name="ir"][data-value]')
        if rating_element is None:
            rating_element = tr.select_one('[itemprop="ratingValue"]')

        votes_element = tr.select_one('strong[name="nv"]')
        if votes_element is None:
            votes_element = tr.select_one('[itemprop="ratingCount"]')

        rows.append({
            "snapshot": snapshot,
            "rank": rank,
            "title": title_link.get_text(" ", strip=True),
            "year": year_match.group() if year_match else "",
            "imdb_id": title_id(title_url),
            "title_url": title_url,
            "rating": (
                rating_element.get("data-value")
                or rating_element.get("content")
                or rating_element.get_text(" ", strip=True)
                if rating_element else ""
            ),
            "votes": (
                votes_element.get("data-value")
                or votes_element.get("content")
                or votes_element.get_text(" ", strip=True)
                if votes_element else ""
            ),
            "source_file": str(source),
        })

    return sorted(rows, key=lambda row: int(row["rank"]))[:250]

def convert(source):
    html = source.read_text(encoding="utf-8", errors="replace")
    soup = BeautifulSoup(html, "html.parser")

    rows = parse_json_ld(soup, source) or parse_table_rows(soup, source)
    if not rows:
        print(f"WARNING: no chart rows recognized; skipped {source}")
        return

    output = source.with_suffix(".csv")
    with output.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=FIELDS)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Wrote {len(rows)} rows: {output}")


for folder in sorted(Path(".").glob("*_top_250")):
    for html_file in sorted(folder.glob("*_top_250.html")):
        convert(html_file)

Wrote 250 rows: 2005_top_250/20050122043755_top_250.csv
Wrote 250 rows: 2005_top_250/20050127043219_top_250.csv
Wrote 250 rows: 2005_top_250/20050131025007_top_250.csv
Wrote 250 rows: 2005_top_250/20050228222121_top_250.csv
Wrote 250 rows: 2005_top_250/20050331093358_top_250.csv
Wrote 250 rows: 2005_top_250/20050422002056_top_250.csv
Wrote 250 rows: 2005_top_250/20050531010815_top_250.csv
Wrote 250 rows: 2005_top_250/20050626025239_top_250.csv
Wrote 250 rows: 2005_top_250/20050727083819_top_250.csv
Wrote 250 rows: 2005_top_250/20050831092654_top_250.csv
Wrote 250 rows: 2005_top_250/20050911141212_top_250.csv
Wrote 250 rows: 2005_top_250/20051030061127_top_250.csv
Wrote 250 rows: 2005_top_250/20051113181046_top_250.csv
Wrote 250 rows: 2005_top_250/20051224015324_top_250.csv
Wrote 250 rows: 2006_top_250/20060116084921_top_250.csv
Wrote 250 rows: 2006_top_250/20060207034420_top_250.csv
Wrote 250 rows: 2006_top_250/20060306214946_top_250.csv
Wrote 250 rows: 2006_top_250/20060422215140_top_